In [0]:
# Read bronze tables
cards_bronze_df = spark.read.table("jarvis_training.bronze.cards_data_bronze")
users_bronze_df = spark.read.table("jarvis_training.bronze.users_data_bronze")
transactions_bronze_df = spark.read.table("jarvis_training.bronze.transactions_data_bronze")
mcc_bronze_df = spark.read.table("jarvis_training.bronze.mcc_codes_bronze")
train_fraud_bronze_df = spark.read.table("jarvis_training.bronze.train_fraud_labels_bronze")

In [0]:
# Cast and clean cards_bronze_df bronze table
from pyspark.sql.functions import *

cards_silver_df = (
    cards_bronze_df
    .select(
        col("id"),
        col("client_id"),
        col("card_brand"),
        col("card_type"),
        col("card_number"),
        col("expires"),
        col("cvv"),
        col("has_chip").alias("has_chip_raw"),
        col("num_cards_issued"),
        regexp_replace(col("credit_limit"), r"[$,]", "").cast("double").alias("credit_limit"),
        to_date(concat(lit("01/"), col("acct_open_date")), "dd/MM/yyyy").alias("acct_open_date"),
        col("year_pin_last_changed").cast("int").alias("year_pin_last_changed"),
        regexp_replace(lower(col("has_chip")), r"[^a-z]", "").alias("has_chip_letters"),
        regexp_replace(lower(col("card_on_dark_web")), r"[^a-z]", "").alias("card_on_dark_web_letters")
    )
    .withColumn(
        "has_chip",
        when(col("has_chip_letters") == "yes", lit(True))
        .when(col("has_chip_letters") == "no", lit(False))
        .otherwise(None)
    )
    .withColumn(
        "card_on_dark_web",
        when(col("card_on_dark_web_letters") == "yes", lit(True))
        .when(col("card_on_dark_web_letters") == "no", lit(False))
        .otherwise(None)
    )
    .drop("has_chip_letters", "card_on_dark_web_letters")
    .dropDuplicates(["id"])
)

display(cards_silver_df.limit(20))

cards_silver_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_training.silver.cards_data_silver")

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip_raw,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,has_chip,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,24295.0,2002-09-01,2008,true,false
2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,21968.0,2014-04-01,2014,true,false
3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,46414.0,2003-07-01,2004,true,false
42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,12400.0,2003-01-01,2012,false,false
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,28.0,2008-09-01,2009,true,false
4537,1746,Visa,Credit,4404898874682993,09/2003,736,YES,1,27500.0,2003-09-01,2012,true,false
1278,1746,Visa,Debit,4001482973848631,07/2022,972,YES,2,28508.0,2011-02-01,2011,true,false
3687,1746,Mastercard,Debit,5627220683410948,06/2022,48,YES,2,9022.0,2003-07-01,2015,true,false
3465,1746,Mastercard,Debit (Prepaid),5711382187309326,11/2020,722,YES,2,54.0,2010-06-01,2015,true,false
3754,1746,Mastercard,Debit (Prepaid),5766121508358701,02/2023,908,YES,1,99.0,2006-07-01,2012,true,false


In [0]:
# Cast and clean mcc_bronze_df bronze table
mcc_silver_df = (
    mcc_bronze_df
    .select(
        col("mcc_code"),
        col("mcc_description")
    )
    .dropDuplicates(["mcc_code"])
)

display(mcc_silver_df.limit(10))

mcc_silver_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_training.silver.mcc_codes_silver")

mcc_code,mcc_description
5812,Eating Places and Restaurants
5541,Service Stations
7996,"Amusement Parks, Carnivals, Circuses"
5411,"Grocery Stores, Supermarkets"
4784,Tolls and Bridge Fees
4900,"Utilities - Electric, Gas, Water, Sanitary"
5942,Book Stores
5814,Fast Food Restaurants
4829,Money Transfer
5311,Department Stores


In [0]:
#  Cast and clean train_fraud_labels bronze table
train_fraud_silver_df = (
    train_fraud_bronze_df
    .select(
        col("transaction_id").cast("int").alias("transaction_id"),
        col("is_fraud").alias("is_fraud")
    )
    .dropDuplicates(["transaction_id"])
)

display(train_fraud_silver_df.limit(10))

train_fraud_silver_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_training.silver.train_fraud_labels_silver")

transaction_id,is_fraud
10649266,false
23410063,false
9316588,false
12478022,false
9558530,false
12532830,false
19526714,false
9906964,false
13224888,false
13749094,false


In [0]:
#  Cast and clean transcation_data bronze table
transactions_clean_df = (
    transactions_bronze_df
    .select(
        col("id").alias("transaction_id"),
        to_timestamp(col("date")).alias("transaction_ts"),
        to_date(col("date")).alias("transaction_date"),
        col("client_id"),
        col("card_id"),
        regexp_replace(col("amount"), r"[$,]", "").cast("double").alias("amount"),
        col("use_chip"),
        col("merchant_id"),
        col("merchant_city"),
        col("merchant_state"),
        regexp_replace(col("zip").cast("string"), r"\.0$", "").alias("zip"),
        col("mcc").alias("mcc_code"),
        col("errors").alias("errors")
    )
    .withColumn("transaction_year", year("transaction_ts"))
    .withColumn("transaction_month", month("transaction_ts"))
    .withColumn("transaction_day", dayofmonth("transaction_ts"))
    .withColumn("day_of_week_num", dayofweek("transaction_ts"))
    .withColumn("day_of_week", date_format("transaction_ts", "EEEE"))
    .withColumn("hour_of_day", hour("transaction_ts"))
    .withColumn(
        "time_of_day",
        when((col("hour_of_day") >= 5) & (col("hour_of_day") < 12), "Morning")
         .when((col("hour_of_day") >= 12) & (col("hour_of_day") < 17), "Afternoon")
         .when((col("hour_of_day") >= 17) & (col("hour_of_day") < 21), "Evening")
         .otherwise("Night")
    )
    .dropDuplicates(["transaction_id"])
)

# Enrich with fraud labels
transactions_silver_df = (
    transactions_clean_df.alias("t")
    .join(
        train_fraud_silver_df.alias("f"),
        col("t.transaction_id") == col("f.transaction_id"),
        "left"
    )
    .join(
        mcc_silver_df.alias("m"),
        col("t.mcc_code") == col("m.mcc_code"),
        "left"
    )
    .select(
        col("t.transaction_id"),
        col("t.transaction_ts"),
        col("t.transaction_date"),
        col("t.transaction_year"),
        col("t.transaction_month"),
        col("t.transaction_day"),
        col("t.day_of_week_num"),
        col("t.day_of_week"),
        col("t.hour_of_day"),
        col("t.time_of_day"),
        col("t.client_id"),
        col("t.card_id"),
        col("t.amount"),
        col("t.use_chip"),
        col("t.merchant_id"),
        col("t.merchant_city"),
        col("t.merchant_state"),
        col("t.zip"),
        col("t.mcc_code"),
        col("m.mcc_description"),
        col("t.errors"),
        coalesce(col("f.is_fraud"),lit(False)).alias("is_fraud")
    )
)

display(transactions_silver_df.limit(20))

transactions_silver_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_training.silver.transactions_data_silver")

transaction_id,transaction_ts,transaction_date,transaction_year,transaction_month,transaction_day,day_of_week_num,day_of_week,hour_of_day,time_of_day,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc_code,mcc_description,errors,is_fraud
7475432,2010-01-01T02:25:00.000Z,2010-01-01,2010,1,1,6,Friday,2,Night,1214,5508,51.58,Online Transaction,98436,ONLINE,null,null,5192,"Books, Periodicals, Newspapers",null,false
7475564,2010-01-01T05:56:00.000Z,2010-01-01,2010,1,1,6,Friday,5,Morning,530,3344,2.98,Swipe Transaction,88646,Colton,CA,92324,5812,Eating Places and Restaurants,null,false
7475977,2010-01-01T07:46:00.000Z,2010-01-01,2010,1,1,6,Friday,7,Morning,573,3268,7.59,Swipe Transaction,23866,Lincolnton,GA,30817,5812,Eating Places and Restaurants,null,false
7477074,2010-01-01T11:54:00.000Z,2010-01-01,2010,1,1,6,Friday,11,Morning,866,2110,13.28,Swipe Transaction,10126,Pittsburgh,PA,15203,5812,Eating Places and Restaurants,null,false
7477902,2010-01-01T14:49:00.000Z,2010-01-01,2010,1,1,6,Friday,14,Afternoon,450,5176,73.0,Swipe Transaction,50783,Newport News,VA,23606,5411,"Grocery Stores, Supermarkets",null,false
7478488,2010-01-01T17:24:00.000Z,2010-01-01,2010,1,1,6,Friday,17,Evening,804,213,25.61,Swipe Transaction,50939,Bowling Green,KY,42101,5813,Drinking Places (Alcoholic Beverages),null,false
7478765,2010-01-01T19:00:00.000Z,2010-01-01,2010,1,1,6,Friday,19,Evening,1169,5763,35.06,Swipe Transaction,89707,Round Rock,TX,78665,4121,Taxicabs and Limousines,null,false
7478844,2010-01-01T19:34:00.000Z,2010-01-01,2010,1,1,6,Friday,19,Evening,456,4576,27.26,Online Transaction,39021,ONLINE,null,null,4784,Tolls and Bridge Fees,null,false
7479199,2010-01-01T21:43:00.000Z,2010-01-01,2010,1,1,6,Friday,21,Night,241,58,58.86,Swipe Transaction,92883,Rockingham,NC,28379,5812,Eating Places and Restaurants,null,false
7479316,2010-01-01T22:29:00.000Z,2010-01-01,2010,1,1,6,Friday,22,Night,1417,2975,-155.0,Swipe Transaction,39991,Lincoln Park,MI,48146,3771,Railroad Passenger Transport,null,false


In [0]:
#  Cast and clean user_data bronze table

users_silver_df = (
    users_bronze_df
    .select(
        col("id").cast("int").alias("client_id"),
        col("current_age").cast("int").alias("current_age"),
        col("retirement_age").cast("int").alias("retirement_age"),
        col("birth_year").cast("int").alias("birth_year"),
        col("birth_month").cast("int").alias("birth_month"),
        col("gender"),
        col("address"),
        col("latitude").cast("double").alias("latitude"),
        col("longitude").cast("double").alias("longitude"),
        col("per_capita_income").alias("per_capita_income_raw"),
        col("yearly_income").alias("yearly_income_raw"),
        col("total_debt").alias("total_debt_raw"),
        col("credit_score").cast("int").alias("credit_score"),
        col("num_credit_cards").cast("int").alias("num_credit_cards")
    )
    .withColumn(
        "per_capita_income",
        regexp_replace(col("per_capita_income_raw"), r"[$,]", "").cast("double")
    )
    .withColumn(
        "yearly_income",
        regexp_replace(col("yearly_income_raw"), r"[$,]", "").cast("double")
    )
    .withColumn(
        "total_debt",
        regexp_replace(col("total_debt_raw"), r"[$,]", "").cast("double")
    )
    .drop("per_capita_income_raw", "yearly_income_raw", "total_debt_raw")
    .dropDuplicates(["client_id"])
)

display(users_silver_df.limit(10))

users_silver_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_training.silver.users_data_silver")

client_id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,credit_score,num_credit_cards,per_capita_income,yearly_income,total_debt
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,787,5,29278.0,59696.0,127613.0
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,701,5,37891.0,77254.0,191349.0
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,698,5,22681.0,33483.0,196.0
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,722,4,163145.0,249925.0,202328.0
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,675,1,53797.0,109687.0,183855.0
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,704,3,20599.0,41997.0,0.0
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,672,3,25258.0,51500.0,102286.0
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,728,1,26790.0,54623.0,114711.0
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,755,5,26273.0,42509.0,2895.0
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,810,1,18730.0,38190.0,81262.0
